In [32]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
import re

import sys
from collections import defaultdict

In [33]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b',
    '--data', 'gully',
    '--data_path','/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/reason_first_results_llama3.2-vision:90b.json',
    '--subset', 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [34]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 samples it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [35]:
# Load test set:
file_path = os.path.join(args.data_path)
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [36]:
data

{'100': ['/root/home/data/neg_collage_100.jpg',
  'To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are small channels that form on hillslopes due to surface runoff and can be an indicator of soil erosion. They typically appear as narrow, shallow depressions or channels on the landscape.\n\nLooking at the images provided, I notice that there is no visible evidence of ephemeral gullies in any of them. The images show a consistent pattern of vegetation cover with no signs of channelization or erosion features characteristic of ephemeral gullies.\n\nTherefore, based on this analysis, it can be concluded that there are no ephemeral gully appearances in the collage of six images.'],
 '1004': ['/root/home/data/neg_collage_1004.jpg',
  "To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are sma

In [37]:
data_prompt = {}
for key, item in data.items():
    prompt = "\nA:" + item[-1]
    data_prompt[key] = [prompt]

In [38]:
data_prompt

{'100': ['\nA:To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are small channels that form on hillslopes due to surface runoff and can be an indicator of soil erosion. They typically appear as narrow, shallow depressions or channels on the landscape.\n\nLooking at the images provided, I notice that there is no visible evidence of ephemeral gullies in any of them. The images show a consistent pattern of vegetation cover with no signs of channelization or erosion features characteristic of ephemeral gullies.\n\nTherefore, based on this analysis, it can be concluded that there are no ephemeral gully appearances in the collage of six images.'],
 '1004': ["\nA:To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are small channels that form on sloping surfaces due to runoff water and can be te

In [39]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [40]:
count = 0
saving_response=copy.deepcopy(data_prompt)

for key, info in tqdm(data_prompt.items()):

    #print(key)
    #print(info[0])
    #sys.exit()
    question = "Answer only with yes or no, does the following response suggest that there is an ephemeral gully:\n\n"
    question += info[0]
    #question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
    
    #print(question)
    #sys.exit()
        
    count+=1
    #for question in questions:
    response = ollama.generate(model=model_name, 
                               prompt=question, 
                               #images=info, 
                               options=options)
    saving_response[key].append(response['response'])
    print(response['response'])

    #sys.exit()
        

  0%|▎                                                                                                 | 1/311 [03:00<15:34:20, 180.84s/it]

No


  1%|▋                                                                                                   | 2/311 [03:06<6:39:20, 77.54s/it]

Yes


  1%|▉                                                                                                   | 3/311 [03:12<3:50:11, 44.84s/it]

...

No


100%|███████████████████████████████████████████████████████████████████████████████████████████████████▋| 310/311 [30:49<00:05,  5.36s/it]

Yes


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [30:54<00:00,  5.96s/it]

No


In [41]:
saving_response

{'100': ['\nA:To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are small channels that form on hillslopes due to surface runoff and can be an indicator of soil erosion. They typically appear as narrow, shallow depressions or channels on the landscape.\n\nLooking at the images provided, I notice that there is no visible evidence of ephemeral gullies in any of them. The images show a consistent pattern of vegetation cover with no signs of channelization or erosion features characteristic of ephemeral gullies.\n\nTherefore, based on this analysis, it can be concluded that there are no ephemeral gully appearances in the collage of six images.',
  'No'],
 '1004': ["\nA:To determine if there are any ephemeral gully appearances in the collage of six images, we need to analyze each image carefully. \n\nEphemeral gullies are small channels that form on sloping surfaces due to runoff water and c

In [42]:
saving_response_reformat = {}
for key, item in saving_response.items():
    saving_response_reformat[key] = item[-1]
saving_response_reformat

{'100': 'No',
 '1004': 'Yes',
 '1005': 'Yes',
 '1007': 'Yes',
 '1010': 'Yes',
 '1015': 'Yes',
 '1017': 'No',
 '1018': 'No',
 '1020': 'Yes',
 '1021': 'Yes',
 '1031': 'Yes',
 '1032': 'Yes',
 '1035': 'Yes',
 '1038': 'No',
 '104': 'Yes',
 '1040': 'Yes',
 '1041': 'Yes',
 '1043': 'Yes',
 '1049': 'Yes',
 '105': 'Yes',
 '1050': 'Yes',
 '1051': 'Yes',
 '1053': 'Yes',
 '1062': 'Yes',
 '1064': 'Yes',
 '1065': 'Yes',
 '1067': 'No',
 '107': 'No',
 '1073': 'No',
 '1078': 'Yes',
 '1081': 'Yes',
 '1085': 'Yes',
 '1095': 'Yes',
 '1098': 'Yes',
 '1099': 'Yes',
 '1114': 'Yes',
 '1117': 'Yes',
 '1121': 'Yes',
 '1123': 'No',
 '1128': 'Yes',
 '1133': 'No',
 '1137': 'Yes',
 '1138': 'No',
 '1140': 'No',
 '1141': 'Yes',
 '1143': 'Yes',
 '117': 'Yes',
 '118': 'Yes',
 '123': 'No',
 '124': 'Yes',
 '125': 'Yes',
 '126': 'No',
 '128': 'No',
 '129': 'Yes',
 '131': 'Yes',
 '135': 'Yes',
 '248': 'No',
 '250': 'No',
 '251': 'Yes',
 '253': 'Yes',
 '266': 'Yes',
 '267': 'Yes',
 '272': 'No',
 '273': 'Yes',
 '287': 'Yes',


In [43]:
saving_response_final = copy.deepcopy(saving_response_reformat)

for key, item in saving_response_reformat.items():
    if 'yes' in item.lower():
        saving_response_final[key] = ['4']
    if 'no' in item.lower():
        saving_response_final[key] = ['0']

saving_response_final

{'100': ['0'],
 '1004': ['4'],
 '1005': ['4'],
 '1007': ['4'],
 '1010': ['4'],
 '1015': ['4'],
 '1017': ['0'],
 '1018': ['0'],
 '1020': ['4'],
 '1021': ['4'],
 '1031': ['4'],
 '1032': ['4'],
 '1035': ['4'],
 '1038': ['0'],
 '104': ['4'],
 '1040': ['4'],
 '1041': ['4'],
 '1043': ['4'],
 '1049': ['4'],
 '105': ['4'],
 '1050': ['4'],
 '1051': ['4'],
 '1053': ['4'],
 '1062': ['4'],
 '1064': ['4'],
 '1065': ['4'],
 '1067': ['0'],
 '107': ['0'],
 '1073': ['0'],
 '1078': ['4'],
 '1081': ['4'],
 '1085': ['4'],
 '1095': ['4'],
 '1098': ['4'],
 '1099': ['4'],
 '1114': ['4'],
 '1117': ['4'],
 '1121': ['4'],
 '1123': ['0'],
 '1128': ['4'],
 '1133': ['0'],
 '1137': ['4'],
 '1138': ['0'],
 '1140': ['0'],
 '1141': ['4'],
 '1143': ['4'],
 '117': ['4'],
 '118': ['4'],
 '123': ['0'],
 '124': ['4'],
 '125': ['4'],
 '126': ['0'],
 '128': ['0'],
 '129': ['4'],
 '131': ['4'],
 '135': ['4'],
 '248': ['0'],
 '250': ['0'],
 '251': ['4'],
 '253': ['4'],
 '266': ['4'],
 '267': ['4'],
 '272': ['0'],
 '273': ['4']

In [44]:
# Save the dictionary as a JSON file
filename =  'reason_first_final_results_'+args.modelname+'.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response_final, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file reason_first_final_results_llama3.2-vision:90b.json has been created.
